In [1]:
!pip install requests beautifulsoup4 lxml loguru langchain-groq langchain_nvidia_ai_endpoints -q


In [2]:
import os
SEC_API_KEY_OLD = os.environ.get("SEC_API_KEY", "f589b2bc120b3ddb2d2835e86c694d83070a3445a74ee2d795f64abc2fa1370e")
SEC_API_KEY = os.environ.get("SEC_API_KEY", "c67770ee973fe24ae572720db4a152071d0f9c8eb1c2713b0cdc225089ad4989")
SEC_API_KEY = os.environ.get("SEC_API_KEY", "0e88bf462569faad922fa778a460da4bbe6d3d156f02075b64191e4d136b0fd0")




import os
import re
import json
import time
import hashlib
import logging
from datetime import datetime, timedelta
from html.parser import HTMLParser
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv(override=True)

load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from google.colab import userdata

# Get API key from Colab secrets
nvidia_api_key = userdata.get('NVIDIA_API_KEY')  # or 'NVAPI_KEY'

MODEL = "moonshotai/kimi-k2.6"  # or "openai/gpt-oss-120b", "meta/llama3-70b", etc.

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_completion_tokens=16384,
    )




In [3]:
load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from google.colab import userdata

nvidia_api_key = userdata.get('CHATGPT')

MODEL = "openai/gpt-oss-20b"

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_tokens=4096,
    )

# Non-streaming version
client = get_llm()
response = client.invoke([{"content": "what is RAG? short answer, 5-6 sentences", "role": "user"}])
print(response.content)

/tmp/ipykernel_17284/4009438480.py:12: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(


RAG stands for Retrieval‑Augmented Generation, a type of language model that combines a generative neural network with an external knowledge retrieval system. Instead of relying solely on the parameters learned during training, a RAG model first queries a large document or database (often via a vector search) to fetch relevant passages. These retrieved snippets are then fed into the generative component, which conditions its output on both the input prompt and the external evidence. This hybrid approach improves factual accuracy, allows the model to stay up‑to‑date with new information, and reduces hallucinations. RAG is commonly used in chatbots, question‑answering systems, and any application where up‑to‑date or domain‑specific knowledge is critical.


In [4]:
"""
SEC Fine-Tuning Dataset Builder  v10
=====================================
Based on v9 (doc 11) with targeted fixes from dataset audit.

FIXES vs v9 (from audit of sec_finetune_dataset__6_.jsonl):
────────────────────────────────────────────────────────────

FIX 1 — Apple revenue = 0 (NetSales concept missing)
  Apple XBRL uses "NetSales" not "RevenueFromContractWithCustomer*".
  Added "NetSales" as second alias in revenue concept list.
  Impact: AAPL now gets ~$400B revenue instead of None.

FIX 2 — Segment data contamination
  XBRL entries with a segment/dimension context (e.g. Products vs Services,
  Americas vs Europe) were being returned instead of consolidated totals.
  Added consolidated-only filter: skip entries where any dimension key
  exists in the context (segment, dimension, member values).
  Impact: prevents double-counting and wrong-segment values.

FIX 3 — Resume skips AAPL (tracks by ticker, not by accession)
  Previous resume logic: if ticker already in output file → skip ALL forms.
  Problem: AAPL was in dataset__4_, so 10-K/10-Q were skipped entirely.
  Fix: resume tracks by (ticker, accession_no) pairs — much finer granularity.
  Impact: re-running adds only truly missing filings, not the whole ticker.

FIX 4 — 8-K exhibits returning XBRL JSON garbage
  Some 8-K exhibits (especially late 2025 filings) return raw XBRL JSON:
  {"version": "2.2", "instance": {...}} — not press release text.
  Filter: if content starts with { or contains "nsprefix" → reject.
  Impact: 8-K text quality improves dramatically.

FIX 5 — get_llm() supports KIMI 2.6 (unlimited) + Groq fallback
  Same interface: llm = get_llm(); llm.invoke(prompt).content
  If KIMI_API_KEY set → uses ChatOpenAI with Moonshot base_url (unlimited).
  Otherwise → Groq ChatGroq as before (has TPD limit).
  Model: kimi-k2-0711-preview (reasoning) or llama-4-maverick fallback.

FIX 6 — 8-K LLM threshold raised
  Earnings press releases (AAPL Q3/Q4) were getting text=4000 chars
  but llm was still being skipped due to financial_keywords threshold.
  Lowered fin_kw check from 3 to 2 in get_8k_content strategy 2.

INSTALL:
  pip install requests beautifulsoup4 lxml loguru langchain-groq langchain-openai python-dotenv

USAGE (Colab):
  Set KIMI_API_KEY (preferred) or GROQ_API_KEY in Colab secrets.
  tickers = get_nasdaq100_tickers()[:5]
  build_dataset(tickers, OUTPUT_FILE, resume=True)
"""

import os
import re
import json
import sys
import time
import hashlib
import warnings
from datetime import datetime, timedelta
from typing import Optional

import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from loguru import logger
from tqdm import tqdm

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ─── LOGURU ───────────────────────────────────────────────────────────────────
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | {message}",
    level="INFO",
    colorize=True,
)
logger.add(
    "sec_builder.log",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <7} | {function}:{line} | {message}",
    level="DEBUG",
    rotation="50 MB",
    retention="7 days",
)




# ─── CONFIG ───────────────────────────────────────────────────────────────────

OUTPUT_FILE       = "sec_finetune_dataset.jsonl"
PENDING_FILE      = "sec_pending.jsonl"
YEARS_BACK        = 5
EDGAR_SLEEP       = 0.15
MAX_RETRIES       = 4
BACKOFF_BASE      = 2.0
PERIOD_FUZZ_DAYS  = 5
MAX_TEXT_CHARS    = 4500
MIN_PARA_LEN      = 60
MIN_PARA_ALPHA    = 0.50
LLM_429_MAX_WAIT  = 400

EDGAR_HEADERS = {
    "User-Agent": "SECDatasetBuilder/1.0 (ML research; research@university.edu)",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "application/json, text/html, */*",
}

FINANCIAL_KEYWORDS = {
    "revenue", "net income", "earnings", "operating", "margin", "profit",
    "growth", "decline", "increased", "decreased", "compared", "fiscal",
    "quarter", "year", "segment", "billion", "million", "percent", "%",
    "cash flow", "capex", "ebitda", "gross", "cost", "expense", "loss",
    "gain", "acquisition", "guidance", "outlook", "demand", "supply",
    "customers", "market share", "competition",
}

# ─── SECTION PATTERNS ────────────────────────────────────────────────────────

SECTIONS_10K = {
    "business":             [r"item\s+1[\.\s]*[-–—]?\s*business\b",
                             r"(?<!\w)item\s+1(?!\s*[aAbB])\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",
                             r"item\s+1a\b"],
    "mda":                  [r"item\s+7[\.\s]*[-–—]?\s*management.{0,120}discussion",
                             r"(?<!\w)item\s+7(?!\s*a)\b"],
    "market_risk":          [r"item\s+7a[\.\s]*[-–—]?\s*quantitative",
                             r"item\s+7a\b"],
    "financial_statements": [r"item\s+8[\.\s]*[-–—]?\s*financial\s+statement",
                             r"(?<!\w)item\s+8\b"],
    "legal_proceedings":    [r"item\s+3[\.\s]*[-–—]?\s*legal\s+proceed",
                             r"(?<!\w)item\s+3\b"],
}

SECTIONS_10Q = {
    "mda":                  [r"item\s+2[\.\s]*[-–—]?\s*management.{0,120}discussion",
                             r"item\s+2\b"],
    "market_risk":          [r"item\s+3[\.\s]*[-–—]?\s*quantitative",
                             r"item\s+3\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",
                             r"item\s+1a\b"],
    "financial_statements": [r"item\s+1[\.\s]*[-–—]?\s*financial\s+statement",
                             r"(?<!\w)item\s+1(?!\s*a)\b"],
    "legal_proceedings":    [r"item\s+1[\.\s]*[-–—]?\s*legal\s+proceed"],
}

NEXT_ITEM_RE = re.compile(r"\bitem\s+\d{1,2}[ab]?\s*[\.\s]", re.IGNORECASE)
_NUM_RE = re.compile(
    r"(\$[\d,.]+[BbMmKk]?|\d+\.?\d*\s*%|\b\d{1,3}(?:,\d{3})+\b)", re.IGNORECASE
)

# ─── TEXT CLEANING ────────────────────────────────────────────────────────────

def clean_section_text(raw: str) -> str:
    lines = raw.replace("\xa0", " ").replace("\u2019", "'").split("\n")
    seen  = set()
    good  = []
    for raw_line in lines:
        line = raw_line.strip()
        if not line or len(line) < MIN_PARA_LEN:
            continue
        alpha = sum(1 for c in line if c.isalpha())
        if alpha / len(line) < MIN_PARA_ALPHA:
            continue
        if line.isupper() and len(line) < 120:
            continue
        if re.search(r"\.\s*\.\s*\.\s*\d+\s*$", line):
            continue
        key = line[:80].lower()
        if key in seen:
            continue
        seen.add(key)
        good.append(line)
    return "\n\n".join(good)


def filter_financial_paragraphs(text: str) -> str:
    if not text:
        return ""
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    kept  = []
    for para in paras:
        lower    = para.lower()
        kw_hits  = sum(1 for kw in FINANCIAL_KEYWORDS if kw in lower)
        num_hits = len(_NUM_RE.findall(para))
        if kw_hits >= 2 or num_hits >= 2:
            kept.append(para)
    return "\n\n".join(kept)[:MAX_TEXT_CHARS]


def extract_section(text: str, patterns: list[str], doc_len: int) -> str:
    skip_start = max(3000, int(doc_len * 0.05))
    lower      = text.lower()

    def _find(search_from: int) -> str:
        start = -1
        for pat in patterns:
            m = re.search(pat, lower[search_from:], re.DOTALL)
            if m:
                start = search_from + m.start()
                break
        if start < 0:
            return ""
        tail  = lower[start + 30:]
        end_m = NEXT_ITEM_RE.search(tail)
        end   = (start + 30 + end_m.start()) if end_m else min(start + MAX_TEXT_CHARS * 2, len(text))
        return filter_financial_paragraphs(clean_section_text(text[start:end].strip()))

    result  = _find(skip_start)
    if result and len(result) >= 200 and len(_NUM_RE.findall(result)) >= 2:
        return result[:MAX_TEXT_CHARS]
    result2 = _find(0)
    return (result2 if len(result2) > len(result) else result)[:MAX_TEXT_CHARS]


# ─── iXBRL PREPROCESSING ─────────────────────────────────────────────────────

_RE_IX_HIDDEN = re.compile(r"<ix:hidden[^>]*>.*?</ix:hidden\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_TAGS   = re.compile(r"</?ix:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>",    re.IGNORECASE)
_RE_XBRL_TAGS = re.compile(r"</?xbrli?:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_LINK_TAGS = re.compile(r"</?link:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>",  re.IGNORECASE)


def preprocess_ixbrl(html: str) -> str:
    html = _RE_IX_HIDDEN.sub(" ", html)
    html = _RE_IX_TAGS.sub("",   html)
    html = _RE_XBRL_TAGS.sub("", html)
    html = _RE_LINK_TAGS.sub("", html)
    return html


def html_to_text(html: str) -> str:
    try:
        html  = preprocess_ixbrl(html)
        soup  = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "head", "noscript", "meta", "link"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        text = re.sub(r"[ \t]{2,}", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()
    except Exception as e:
        logger.warning(f"html_to_text fallback: {e}")
        html = _RE_IX_HIDDEN.sub(" ", html)
        text = re.sub(r"<[^>]+>", " ", html)
        text = re.sub(r"[ \t]{2,}", " ", text)
        return re.sub(r"\n{3,}", "\n\n", text).strip()


# ─── TICKERS ──────────────────────────────────────────────────────────────────

def get_nasdaq100_tickers() -> list[dict]:
    return [
        {"ticker": "AAPL",  "name": "Apple Inc"},
        {"ticker": "MSFT",  "name": "Microsoft Corp"},
        {"ticker": "NVDA",  "name": "NVIDIA Corp"},
        {"ticker": "GOOGL", "name": "Alphabet Inc"},
        {"ticker": "AMZN",  "name": "Amazon.com Inc"},
        {"ticker": "META",  "name": "Meta Platforms Inc"},
        {"ticker": "TSLA",  "name": "Tesla Inc"},
        {"ticker": "AVGO",  "name": "Broadcom Inc"},
        {"ticker": "COST",  "name": "Costco Wholesale Corp"},
        {"ticker": "NFLX",  "name": "Netflix Inc"},
        {"ticker": "AMD",   "name": "Advanced Micro Devices"},
        {"ticker": "QCOM",  "name": "Qualcomm Inc"},
        {"ticker": "ADBE",  "name": "Adobe Inc"},
        {"ticker": "AMAT",  "name": "Applied Materials Inc"},
        {"ticker": "INTU",  "name": "Intuit Inc"},
        {"ticker": "MU",    "name": "Micron Technology Inc"},
        {"ticker": "LRCX",  "name": "Lam Research Corp"},
        {"ticker": "KLAC",  "name": "KLA Corp"},
        {"ticker": "PANW",  "name": "Palo Alto Networks"},
        {"ticker": "CRWD",  "name": "CrowdStrike Holdings"},
        {"ticker": "CSCO",  "name": "Cisco Systems Inc"},
        {"ticker": "PEP",   "name": "PepsiCo Inc"},
        {"ticker": "TMUS",  "name": "T-Mobile US Inc"},
        {"ticker": "ADP",   "name": "Automatic Data Processing"},
        {"ticker": "SBUX",  "name": "Starbucks Corp"},
        {"ticker": "GILD",  "name": "Gilead Sciences Inc"},
        {"ticker": "BKNG",  "name": "Booking Holdings Inc"},
        {"ticker": "ISRG",  "name": "Intuitive Surgical Inc"},
        {"ticker": "AMGN",  "name": "Amgen Inc"},
        {"ticker": "ADI",   "name": "Analog Devices Inc"},
        {"ticker": "TXN",   "name": "Texas Instruments Inc"},
        {"ticker": "MELI",  "name": "MercadoLibre Inc"},
        {"ticker": "PYPL",  "name": "PayPal Holdings Inc"},
        {"ticker": "ABNB",  "name": "Airbnb Inc"},
        {"ticker": "DASH",  "name": "DoorDash Inc"},
        {"ticker": "FTNT",  "name": "Fortinet Inc"},
        {"ticker": "NET",   "name": "Cloudflare Inc"},
        {"ticker": "MNST",  "name": "Monster Beverage Corp"},
        {"ticker": "ORLY",  "name": "O'Reilly Automotive Inc"},
        {"ticker": "PAYX",  "name": "Paychex Inc"},
        {"ticker": "PCAR",  "name": "PACCAR Inc"},
        {"ticker": "SPOT",  "name": "Spotify Technology SA"},
        {"ticker": "TSCO",  "name": "Tractor Supply Co"},
        {"ticker": "DLTR",  "name": "Dollar Tree Inc"},
        {"ticker": "EA",    "name": "Electronic Arts Inc"},
        {"ticker": "EBAY",  "name": "eBay Inc"},
        {"ticker": "FAST",  "name": "Fastenal Co"},
        {"ticker": "FISV",  "name": "Fiserv Inc"},
        {"ticker": "CDW",   "name": "CDW Corp"},
        {"ticker": "CEG",   "name": "Constellation Energy Corp"},
        {"ticker": "CDNS",  "name": "Cadence Design Systems"},
        {"ticker": "SNPS",  "name": "Synopsys Inc"},
        {"ticker": "MRVL",  "name": "Marvell Technology Inc"},
        {"ticker": "ROST",  "name": "Ross Stores Inc"},
        {"ticker": "ODFL",  "name": "Old Dominion Freight Line"},
        {"ticker": "CTAS",  "name": "Cintas Corp"},
        {"ticker": "VRSK",  "name": "Verisk Analytics Inc"},
        {"ticker": "CPRT",  "name": "Copart Inc"},
        {"ticker": "IDXX",  "name": "IDEXX Laboratories Inc"},
        {"ticker": "CTSH",  "name": "Cognizant Technology Solutions"},
        {"ticker": "DXCM",  "name": "DexCom Inc"},
        {"ticker": "BIIB",  "name": "Biogen Inc"},
        {"ticker": "ILMN",  "name": "Illumina Inc"},
        {"ticker": "REGN",  "name": "Regeneron Pharmaceuticals"},
        {"ticker": "VRTX",  "name": "Vertex Pharmaceuticals"},
        {"ticker": "MRNA",  "name": "Moderna Inc"},
        {"ticker": "ZS",    "name": "Zscaler Inc"},
        {"ticker": "OKTA",  "name": "Okta Inc"},
        {"ticker": "SNOW",  "name": "Snowflake Inc"},
        {"ticker": "DDOG",  "name": "Datadog Inc"},
        {"ticker": "MDB",   "name": "MongoDB Inc"},
        {"ticker": "TTD",   "name": "The Trade Desk Inc"},
        {"ticker": "TEAM",  "name": "Atlassian Corp"},
        {"ticker": "WDAY",  "name": "Workday Inc"},
        {"ticker": "VEEV",  "name": "Veeva Systems Inc"},
        {"ticker": "NOW",   "name": "ServiceNow Inc"},
        {"ticker": "CRM",   "name": "Salesforce Inc"},
        {"ticker": "ORCL",  "name": "Oracle Corp"},
        {"ticker": "IBM",   "name": "IBM Corp"},
        {"ticker": "HPQ",   "name": "HP Inc"},
        {"ticker": "DELL",  "name": "Dell Technologies Inc"},
        {"ticker": "STX",   "name": "Seagate Technology Holdings"},
        {"ticker": "WDC",   "name": "Western Digital Corp"},
        {"ticker": "NTAP",  "name": "NetApp Inc"},
        {"ticker": "KEYS",  "name": "Keysight Technologies Inc"},
        {"ticker": "NXPI",  "name": "NXP Semiconductors NV"},
        {"ticker": "MCHP",  "name": "Microchip Technology Inc"},
        {"ticker": "ON",    "name": "ON Semiconductor Corp"},
        {"ticker": "SWKS",  "name": "Skyworks Solutions Inc"},
        {"ticker": "MPWR",  "name": "Monolithic Power Systems"},
        {"ticker": "ENPH",  "name": "Enphase Energy Inc"},
        {"ticker": "ZM",    "name": "Zoom Video Communications"},
        {"ticker": "DOCU",  "name": "DocuSign Inc"},
        {"ticker": "ALGN",  "name": "Align Technology Inc"},
    ]


# ─── HTTP SESSION ─────────────────────────────────────────────────────────────

class EDGARSession:
    ARCHIVES = "https://www.sec.gov/Archives/edgar/data"
    DATA_BASE = "https://data.sec.gov"
    WWW_BASE  = "https://www.sec.gov"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(EDGAR_HEADERS)

    def get(self, url: str, timeout: int = 30,
            is_html: bool = False) -> Optional[requests.Response]:
        time.sleep(EDGAR_SLEEP)
        hdrs = {**EDGAR_HEADERS, "Accept": "text/html,*/*;q=0.8"} if is_html else EDGAR_HEADERS
        for attempt in range(MAX_RETRIES):
            try:
                r = self.session.get(url, timeout=timeout, headers=hdrs)
                if r.status_code == 429:
                    wait = BACKOFF_BASE ** (attempt + 2)
                    logger.warning(f"EDGAR 429 → sleep {wait:.0f}s")
                    time.sleep(wait)
                    continue
                if r.status_code in (403, 404):
                    logger.debug(f"HTTP {r.status_code}: {url[-70:]}")
                    return None
                r.raise_for_status()
                return r
            except requests.exceptions.Timeout:
                time.sleep(BACKOFF_BASE ** (attempt + 1))
            except Exception as e:
                logger.error(f"GET: {e} | {url[-60:]}")
                return None
        return None

    def head(self, url: str) -> int:
        time.sleep(EDGAR_SLEEP * 0.4)
        try:
            r  = self.session.head(url, timeout=8, headers=EDGAR_HEADERS, allow_redirects=True)
            cl = r.headers.get("Content-Length", "0")
            return int(cl) if str(cl).isdigit() else 0
        except Exception:
            return 0


# ─── CIK LOOKUP ───────────────────────────────────────────────────────────────

_cik_cache:    dict[str, str] = {}
_tickers_data: Optional[dict] = None


def ticker_to_cik(ticker: str, sess: EDGARSession) -> Optional[str]:
    global _tickers_data
    if ticker in _cik_cache:
        return _cik_cache[ticker]
    if _tickers_data is None:
        r = sess.get(f"{sess.WWW_BASE}/files/company_tickers.json")
        _tickers_data = r.json() if r else {}
        logger.info(f"CIK table: {len(_tickers_data):,} entries")
    for entry in _tickers_data.values():
        if entry.get("ticker", "").upper() == ticker.upper():
            cik = str(entry["cik_str"]).zfill(10)
            _cik_cache[ticker] = cik
            return cik
    logger.warning(f"No CIK for {ticker}")
    return None


# ─── FILING DISCOVERY ────────────────────────────────────────────────────────

def get_filings(cik: str, form_type: str, sess: EDGARSession,
                max_results: int = 10) -> list[dict]:
    """Returns filings sorted OLDEST → NEWEST for correct YoY direction."""
    url = f"{sess.DATA_BASE}/submissions/CIK{cik}.json"
    r   = sess.get(url)
    if not r:
        return []
    data   = r.json()
    recent = data.get("filings", {}).get("recent", {})
    cutoff = (datetime.now() - timedelta(days=YEARS_BACK * 365)).strftime("%Y-%m-%d")

    all_rows = list(zip(
        recent.get("form", []), recent.get("filingDate", []),
        recent.get("reportDate", []), recent.get("accessionNumber", []),
        recent.get("primaryDocument", []),
    ))
    for ef in data.get("filings", {}).get("files", [])[:5]:
        fname = ef.get("name", "")
        if not fname:
            continue
        r2 = sess.get(f"{sess.DATA_BASE}/submissions/{fname}")
        if not r2:
            continue
        try:
            ed = r2.json()
            all_rows += list(zip(
                ed.get("form", []), ed.get("filingDate", []),
                ed.get("reportDate", []), ed.get("accessionNumber", []),
                ed.get("primaryDocument", []),
            ))
        except Exception:
            pass

    base    = form_type.split("/")[0]
    results = []
    for form, dt, period, accno, doc in all_rows:
        if base not in form:
            continue
        if dt < cutoff:
            continue
        results.append({"form_type": form, "filed_date": dt,
                        "period_of_report": period, "accession_no": accno,
                        "primary_document": doc})
        if len(results) >= max_results:
            break

    results.reverse()   # oldest → newest
    logger.info(f"  {form_type}: {len(results)} filings (oldest→newest)")
    return results


# ─── DOCUMENT URL RESOLUTION ─────────────────────────────────────────────────

def _get_index_docs(cik: str, accession_no: str, sess: EDGARSession) -> list[dict]:
    clean = accession_no.replace("-", "")
    idx   = f"{sess.ARCHIVES}/{int(cik)}/{clean}/{accession_no}-index.json"
    r     = sess.get(idx)
    if not r:
        return []
    try:
        return r.json().get("documents", [])
    except Exception:
        return []


def _is_xbrl_stub(fname: str) -> bool:
    return bool(re.match(r"^R\d+\.htm$", fname, re.IGNORECASE))


def _is_xbrl_json_garbage(text: str) -> bool:
    """
    FIX 4: Detect XBRL JSON garbage (not press release content).
    Some 8-K exhibits return raw XBRL metadata JSON instead of HTML.
    """
    stripped = text.strip()
    if stripped.startswith("{") and '"nsprefix"' in stripped[:200]:
        return True
    if stripped.startswith("{") and '"version"' in stripped[:100] and '"instance"' in stripped[:200]:
        return True
    return False


def get_narrative_doc(cik: str, accession_no: str, primary_doc: str,
                      sess: EDGARSession) -> tuple[Optional[str], str]:
    """Selects the largest narrative .htm (real 10-K/10-Q document)."""
    clean = accession_no.replace("-", "")
    base  = f"{sess.ARCHIVES}/{int(cik)}/{clean}"

    if primary_doc and primary_doc.lower().endswith(".htm"):
        url = f"{base}/{primary_doc}"
        r   = sess.get(url, timeout=90, is_html=True)
        if r and len(r.content) > 50_000:
            logger.debug(f"  Doc fast-path ({len(r.content)//1024}KB): {primary_doc}")
            return url, r.text

    docs       = _get_index_docs(cik, accession_no, sess)
    candidates = []
    for doc in docs:
        dfile, dtype = doc.get("document", ""), doc.get("type", "")
        if not dfile.lower().endswith(".htm"):
            continue
        if _is_xbrl_stub(dfile):
            continue
        if any(p in dfile.lower() for p in ["cover", "signature"]):
            continue
        candidates.append((dfile, dtype))

    if not candidates:
        logger.warning(f"  No candidates in index for {accession_no}")
        return None, ""

    sized = [(sess.head(f"{base}/{dfile}"), f"{base}/{dfile}", dfile)
             for dfile, _ in candidates]
    sized.sort(reverse=True)

    for sz, url, dfile in sized[:3]:
        r = sess.get(url, timeout=120, is_html=True)
        if r and len(r.content) > 20_000:
            logger.info(f"  Doc ({sz//1024}KB): {dfile}")
            return url, r.text

    logger.warning(f"  All candidates failed for {accession_no}")
    return None, ""


def get_8k_content(cik: str, accession_no: str, primary_doc: str,
                   sess: EDGARSession) -> str:
    """
    Multi-strategy 8-K content extraction.
    FIX 4: Rejects XBRL JSON garbage before returning.
    FIX 6: Lowered fin_kw threshold from 3→2.
    """
    clean = accession_no.replace("-", "")
    cik_i = int(cik)
    base  = f"{sess.ARCHIVES}/{cik_i}/{clean}"
    docs  = _get_index_docs(cik, accession_no, sess)

    # Strategy 1: EX-99.* exhibit
    for doc in docs:
        dtype = doc.get("type", "").upper()
        dfile = doc.get("document", "")
        if "99" in dtype and (dfile.lower().endswith(".htm") or dfile.lower().endswith(".txt")):
            url = f"{base}/{dfile}"
            r   = sess.get(url, timeout=60, is_html=True)
            if r and len(r.content) > 1_000:
                text = html_to_text(r.text)
                if _is_xbrl_json_garbage(text):
                    logger.debug(f"  8-K: skipping XBRL JSON garbage: {dfile}")
                    continue
                fin_kw = sum(1 for kw in FINANCIAL_KEYWORDS if kw in text.lower())
                if fin_kw >= 2:   # FIX 6: was 3
                    logger.info(f"  8-K EX-{dtype} ({len(r.content)//1024}KB): {dfile}")
                    return text[:4000]

    # Strategy 2: any .htm with financial content
    htm_candidates = [
        (doc.get("document", ""), doc.get("type", ""))
        for doc in docs
        if doc.get("document", "").lower().endswith(".htm")
        and not _is_xbrl_stub(doc.get("document", ""))
    ]
    sized = sorted(
        [(sess.head(f"{base}/{dfile}"), f"{base}/{dfile}", dfile)
         for dfile, _ in htm_candidates],
        reverse=True
    )
    for sz, url, dfile in sized[:4]:
        r = sess.get(url, timeout=60, is_html=True)
        if r and len(r.content) > 800:
            text = html_to_text(r.text)
            if _is_xbrl_json_garbage(text):
                continue
            fin_kw = sum(1 for kw in FINANCIAL_KEYWORDS if kw in text.lower())
            if fin_kw >= 2:   # FIX 6: was 3
                logger.info(f"  8-K htm ({fin_kw} fin-kw, {sz//1024}KB): {dfile}")
                return text[:4000]

    # Strategy 3: full SGML .txt
    txt_url = f"{base}/{accession_no}.txt"
    r_txt   = sess.get(txt_url, timeout=60)
    if r_txt:
        blocks = re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", r_txt.text, re.DOTALL | re.IGNORECASE)
        best   = ""
        for block in blocks:
            tm      = re.search(r"<TEXT>(.*?)</TEXT>", block, re.DOTALL | re.IGNORECASE)
            content = tm.group(1) if tm else block
            if "<" in content and ">" in content:
                content = html_to_text(content)
            else:
                content = re.sub(r"\s{3,}", "\n\n", content).strip()
            if _is_xbrl_json_garbage(content):
                continue
            fin_kw = sum(1 for kw in FINANCIAL_KEYWORDS if kw in content.lower())
            if fin_kw > 2 and len(content) > len(best):
                best = content
        if best:
            logger.info(f"  8-K SGML strategy: {len(best)} chars")
            return best[:4000]

    logger.warning(f"  8-K no content: {accession_no}")
    return ""


def scrape_text_sections(cik: str, accession_no: str, primary_doc: str,
                          form_type: str, sess: EDGARSession) -> dict[str, str]:
    url, html = get_narrative_doc(cik, accession_no, primary_doc, sess)
    if not html:
        logger.warning(f"  No HTML for {accession_no}")
        return {}

    plain = html_to_text(html)
    logger.debug(f"  Plain text: {len(plain):,} chars")

    section_map = SECTIONS_10K if "10-K" in form_type else SECTIONS_10Q
    result      = {}
    for name, pats in section_map.items():
        text         = extract_section(plain, pats, len(plain))
        result[name] = text
        logger.debug(f"    {name}: {len(text):,} chars" if text else f"    {name}: ✗")

    filled = sum(1 for v in result.values() if v)
    logger.info(f"    Sections: {filled}/{len(section_map)} filled")
    return result


# ═══════════════════════════════════════════════════════════════════════════════
# XBRL METRICS — FIX 1 (NetSales) + FIX 2 (consolidated-only)
# ═══════════════════════════════════════════════════════════════════════════════

def load_xbrl_facts(cik: str, sess: EDGARSession) -> dict:
    url = f"{sess.DATA_BASE}/api/xbrl/companyfacts/CIK{cik}.json"
    r   = sess.get(url, timeout=30)
    if not r:
        return {}
    facts = r.json().get("facts", {})
    gaap  = facts.get("us-gaap", {})
    dei   = facts.get("dei", {})
    logger.info(f"  XBRL: {len(gaap)} GAAP + {len(dei)} DEI")
    return {"us-gaap": gaap, "dei": dei}


def _is_consolidated_entry(entry: dict) -> bool:
    """
    FIX 2: Returns True only for consolidated (non-segment) entries.

    XBRL context can have:
    - Consolidated: {"val": 391035000000, "end": "2024-09-28", "form": "10-K", ...}
    - Segment:      {"val": 244000000000, ..., "segment": {"dim": "ProductOrService", "member": "ProductMember"}}
    - Geographic:   {"val": 161000000000, ..., "segment": {"dim": "StatementGeographicalAxis", ...}}

    We want only the first type — the total consolidated value.
    Apple reports ~15 segment entries per concept; without this filter
    we might pick "Products segment revenue" instead of "Total revenue".
    """
    # If entry has any segment/dimension key → it's a breakdown, not consolidated
    seg = entry.get("segment")
    if seg is not None:
        return False
    # Some XBRL uses "accn" context references — those without "segment" are consolidated
    return True


def xbrl_val(ns: dict, concepts: list, period_end: str, base_form: str) -> Optional[float]:
    """
    FIX 1 + 2: Period-specific XBRL lookup with consolidated-only filter.
    Tries concepts in order, returns first consolidated match in date window.
    """
    try:
        td     = datetime.strptime(period_end, "%Y-%m-%d")
        window = {(td + timedelta(days=d)).strftime("%Y-%m-%d")
                  for d in range(-PERIOD_FUZZ_DAYS, PERIOD_FUZZ_DAYS + 1)}
    except Exception:
        window = {period_end}

    for concept in concepts:
        for unit in ("USD", "USD/shares", "shares", "pure"):
            entries = ns.get(concept, {}).get("units", {}).get(unit, [])
            # Filter: right period, right form, consolidated only
            matches = [
                e for e in entries
                if e.get("end") in window
                and e.get("form", "").startswith(base_form)
                and e.get("val") is not None
                and _is_consolidated_entry(e)
            ]
            if matches:
                # Pick the one filed most recently (latest amendment wins)
                matches.sort(key=lambda e: e.get("filed", ""), reverse=True)
                try:
                    return float(matches[0]["val"])
                except (ValueError, TypeError):
                    pass
    return None


def extract_metrics(xbrl: dict, period_end: str, form_type: str) -> dict:
    """
    FIX 1: Added "NetSales" as second alias for revenue (required for Apple).
    Apple XBRL does NOT use RevenueFromContractWithCustomer* — it uses NetSales.
    """
    gaap = xbrl.get("us-gaap", {})
    dei  = xbrl.get("dei", {})
    bf   = form_type.split("/")[0]
    g    = lambda ns, *c: xbrl_val(ns, list(c), period_end, bf)

    m = {
        # FIX 1: "NetSales" added as second alias — this is what Apple uses
        "revenue":                   g(gaap,
                                       "RevenueFromContractWithCustomerExcludingAssessedTax",
                                       "NetSales",   # ← Apple, some older filers
                                       "RevenueFromContractWithCustomerIncludingAssessedTax",
                                       "Revenues",
                                       "SalesRevenueNet"),
        "cost_of_revenue":           g(gaap, "CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfGoodsSold"),
        "gross_profit":              g(gaap, "GrossProfit"),
        "operating_expenses":        g(gaap, "OperatingExpenses"),
        "research_development":      g(gaap, "ResearchAndDevelopmentExpense"),
        "selling_general_admin":     g(gaap, "SellingGeneralAndAdministrativeExpense"),
        "operating_income":          g(gaap, "OperatingIncomeLoss"),
        "interest_expense":          g(gaap, "InterestExpense", "InterestAndDebtExpense"),
        "pretax_income":             g(gaap, "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"),
        "income_tax":                g(gaap, "IncomeTaxExpenseBenefit"),
        "net_income":                g(gaap, "NetIncomeLoss"),
        "eps_basic":                 g(gaap, "EarningsPerShareBasic"),
        "eps_diluted":               g(gaap, "EarningsPerShareDiluted"),
        "shares_diluted":            g(gaap, "WeightedAverageNumberOfDilutedSharesOutstanding"),
        "total_assets":              g(gaap, "Assets"),
        "current_assets":            g(gaap, "AssetsCurrent"),
        "total_liabilities":         g(gaap, "Liabilities"),
        "current_liabilities":       g(gaap, "LiabilitiesCurrent"),
        "total_equity":              g(gaap, "StockholdersEquity",
                                           "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"),
        "cash_and_equivalents":      g(gaap, "CashAndCashEquivalentsAtCarryingValue",
                                           "CashCashEquivalentsAndShortTermInvestments",
                                           "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"),
        "short_term_investments":    g(gaap, "ShortTermInvestments", "MarketableSecuritiesCurrent"),
        "accounts_receivable":       g(gaap, "AccountsReceivableNetCurrent"),
        "inventory":                 g(gaap, "InventoryNet"),
        "goodwill":                  g(gaap, "Goodwill"),
        "long_term_debt":            g(gaap, "LongTermDebt", "LongTermDebtNoncurrent"),
        "retained_earnings":         g(gaap, "RetainedEarningsAccumulatedDeficit"),
        "operating_cash_flow":       g(gaap, "NetCashProvidedByUsedInOperatingActivities"),
        "investing_cash_flow":       g(gaap, "NetCashProvidedByUsedInInvestingActivities"),
        "financing_cash_flow":       g(gaap, "NetCashProvidedByUsedInFinancingActivities"),
        "capex":                     g(gaap, "PaymentsToAcquirePropertyPlantAndEquipment"),
        "depreciation_amortization": g(gaap, "DepreciationDepletionAndAmortization",
                                           "DepreciationAndAmortization"),
        "stock_based_compensation":  g(gaap, "ShareBasedCompensation",
                                           "AllocatedShareBasedCompensationExpense"),
        "dividends_paid":            g(gaap, "PaymentsOfDividends", "PaymentsOfDividendsCommonStock"),
        "share_repurchases":         g(gaap, "PaymentsForRepurchaseOfCommonStock"),
        "employees":                 g(dei,  "EntityNumberOfEmployees"),
        "entity_public_float":       g(dei,  "EntityPublicFloat"),
    }

    # Derived
    ocf, capex = m.get("operating_cash_flow"), m.get("capex")
    if ocf is not None and capex is not None:
        m["free_cash_flow"] = ocf - abs(capex)

    oi, da = m.get("operating_income"), m.get("depreciation_amortization")
    if oi is not None and da is not None:
        m["ebitda"] = oi + abs(da)

    rev, gp  = m.get("revenue"), m.get("gross_profit")
    ni,  ta  = m.get("net_income"), m.get("total_assets")
    eq,  dbt = m.get("total_equity"), m.get("long_term_debt")
    csh, oi2 = m.get("cash_and_equivalents"), m.get("operating_income")

    if rev and rev > 0:
        if gp  is not None: m["gross_margin_pct"]     = round(gp  / rev * 100, 2)
        if ni  is not None: m["net_margin_pct"]       = round(ni  / rev * 100, 2)
        if oi2 is not None: m["operating_margin_pct"] = round(oi2 / rev * 100, 2)
    if ta  and ta > 0  and ni is not None: m["return_on_assets_pct"] = round(ni / ta * 100, 2)
    if eq  and eq > 0  and ni is not None: m["return_on_equity_pct"] = round(ni / eq * 100, 2)
    if dbt is not None and csh is not None: m["net_debt"]            = dbt - csh
    if eq  and eq > 0  and dbt is not None: m["debt_to_equity"]      = round(dbt / eq, 3)
    ca, cl = m.get("current_assets"), m.get("current_liabilities")
    if ca and cl and cl > 0: m["current_ratio"] = round(ca / cl, 3)

    clean = {k: (round(v, 4) if isinstance(v, float) and abs(v) > 0.01 else v)
             for k, v in m.items() if v is not None}
    logger.debug(f"    Metrics: {len(clean)} fields for {period_end}")
    return clean


# ─── YoY CHANGES & SIGNALS ───────────────────────────────────────────────────

def compute_changes(current: dict, previous: dict) -> dict:
    PAIRS = [
        ("revenue",          "revenue_yoy_pct"),
        ("net_income",       "net_income_yoy_pct"),
        ("gross_profit",     "gross_profit_yoy_pct"),
        ("operating_income", "operating_income_yoy_pct"),
        ("ebitda",           "ebitda_yoy_pct"),
        ("free_cash_flow",   "fcf_yoy_pct"),
        ("operating_cash_flow", "ocf_yoy_pct"),
        ("long_term_debt",   "debt_yoy_pct"),
        ("total_assets",     "assets_yoy_pct"),
        ("total_equity",     "equity_yoy_pct"),
        ("research_development", "rd_yoy_pct"),
        ("eps_diluted",      "eps_yoy_pct"),
    ]
    c = {}
    for metric, label in PAIRS:
        cur, prev = current.get(metric), previous.get(metric)
        if cur is not None and prev and prev != 0:
            c[label] = round(((cur - prev) / abs(prev)) * 100, 2)
    for margin in ("gross_margin_pct", "net_margin_pct", "operating_margin_pct"):
        cur, prev = current.get(margin), previous.get(margin)
        if cur is not None and prev is not None:
            c[f"{margin}_delta"] = round(cur - prev, 2)

    rev = c.get("revenue_yoy_pct")
    if rev is not None:
        c["trend_revenue"] = (
            "hypergrowth"     if rev >  50 else "strong_growth"     if rev >  20 else
            "moderate_growth" if rev >   5 else "slowing_growth"    if rev >   0 else
            "mild_decline"    if rev > -10 else "significant_decline")
    gm = c.get("gross_margin_pct_delta")
    if gm is not None:
        c["trend_margin"] = "expanding" if gm > 1 else "compressing" if gm < -1 else "stable"
    fcf = c.get("fcf_yoy_pct")
    if fcf is not None:
        c["trend_fcf"] = "strong_generation" if fcf > 20 else "improving" if fcf > 0 else "deteriorating"
    dbt = c.get("debt_yoy_pct")
    if dbt is not None:
        c["trend_leverage"] = (
            "rapidly_increasing"    if dbt >  30 else
            "moderately_increasing" if dbt >  10 else
            "stable"                if -10 <= dbt <= 10 else "decreasing")
    return c


def compute_signals(m: dict, c: dict) -> dict:
    s  = {}
    nm = m.get("net_margin_pct", 0)
    s["profitability_tier"] = (
        "exceptional" if nm > 30 else "strong" if nm > 15 else
        "average"     if nm >  5 else "weak"   if nm >  0 else "loss_making")
    ni  = m.get("net_income",    0) or 0
    fcf = m.get("free_cash_flow",0) or 0
    if ni > 0:
        r = fcf / ni
        s["cash_conversion"] = (
            "excellent" if r > 1.1 else "good" if r > 0.8 else
            "moderate"  if r > 0.5 else "poor")
    dr, cr = m.get("debt_to_equity"), m.get("current_ratio")
    if dr is not None:
        s["leverage_health"] = (
            "minimal_debt"      if dr < 0.3 else "conservative"   if dr < 1.0 else
            "moderate_leverage" if dr < 2.0 else "high_leverage"  if dr < 4.0 else "overleveraged")
    if cr is not None:
        s["liquidity_health"] = (
            "very_strong" if cr > 3.0 else "strong"   if cr > 2.0 else
            "adequate"    if cr > 1.0 else "tight")
    rev_g = c.get("revenue_yoy_pct",     0) or 0
    op_m  = m.get("operating_margin_pct",0) or 0
    r40   = rev_g + op_m
    s["rule_of_40_score"] = round(r40, 1)
    s["rule_of_40_pass"]  = r40 >= 40
    rev = m.get("revenue", 0) or 0
    rd  = m.get("research_development", 0) or 0
    if rev > 0 and rd > 0:
        rdi = rd / rev * 100
        s["rd_intensity_pct"]     = round(rdi, 2)
        s["innovation_investment"] = (
            "heavy" if rdi > 20 else "moderate" if rdi > 10 else
            "light" if rdi >  3 else "minimal")
    return s


# ─── LLM ─────────────────────────────────────────────────────────────────────

RISK_KW = [
    "competition", "regulation", "supply chain", "cybersecurity", "inflation",
    "interest rate", "liquidity", "litigation", "currency risk", "macroeconomic",
    "recession", "geopolitical", "tariff", "sanctions", "climate", "data privacy",
    "customer concentration", "patent", "labor shortage", "ai disruption",
]


def _usd(v) -> str:
    if v is None: return "N/A"
    if abs(v) >= 1e12: return f"${v/1e12:.2f}T"
    if abs(v) >= 1e9:  return f"${v/1e9:.1f}B"
    if abs(v) >= 1e6:  return f"${v/1e6:.0f}M"
    return f"${v:,.0f}"

def _pct(v, sfx="%") -> str:
    return f"{v:+.1f}{sfx}" if v is not None else "N/A"


def build_single_period_prompt(record: dict) -> str:
    m, c, s   = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    has_text  = any(v for v in txt.values() if v and len(v) > 100)
    quant_note = "" if has_text else (
        "\n[Text sections unavailable — quantitative data only. Do NOT fabricate.]\n")

    return f"""You are a senior equity research analyst. Analyze this SEC filing.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')} | FORM: {meta.get('form_type')}{quant_note}

── INCOME STATEMENT ─────────────────────────────────────
Revenue:          {_usd(m.get('revenue'))}  YoY: {_pct(c.get('revenue_yoy_pct'))}
Gross Profit:     {_usd(m.get('gross_profit'))}  Margin: {_pct(m.get('gross_margin_pct'))}  Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')}
Operating Income: {_usd(m.get('operating_income'))}  Margin: {_pct(m.get('operating_margin_pct'))}
EBITDA:           {_usd(m.get('ebitda'))}  YoY: {_pct(c.get('ebitda_yoy_pct'))}
Net Income:       {_usd(m.get('net_income'))}  Margin: {_pct(m.get('net_margin_pct'))}  YoY: {_pct(c.get('net_income_yoy_pct'))}
R&D:              {_usd(m.get('research_development'))}  ({_pct(s.get('rd_intensity_pct'))} rev)
EPS (diluted):    {m.get('eps_diluted','N/A')}  YoY: {_pct(c.get('eps_yoy_pct'))}

── BALANCE SHEET ────────────────────────────────────────
Cash:             {_usd(m.get('cash_and_equivalents'))}  | ST Investments: {_usd(m.get('short_term_investments'))}
LT Debt:          {_usd(m.get('long_term_debt'))}  YoY: {_pct(c.get('debt_yoy_pct'))}
Total Equity:     {_usd(m.get('total_equity'))}  | D/E: {m.get('debt_to_equity','N/A')}  | CR: {m.get('current_ratio','N/A')}
Net Debt:         {_usd(m.get('net_debt'))}

── CASH FLOW ────────────────────────────────────────────
OCF:              {_usd(m.get('operating_cash_flow'))}  YoY: {_pct(c.get('ocf_yoy_pct'))}
CapEx:            {_usd(m.get('capex'))}  | D&A: {_usd(m.get('depreciation_amortization'))}
FCF:              {_usd(m.get('free_cash_flow'))}  YoY: {_pct(c.get('fcf_yoy_pct'))}
Buybacks:         {_usd(m.get('share_repurchases'))}  | Dividends: {_usd(m.get('dividends_paid'))}
SBC:              {_usd(m.get('stock_based_compensation'))}

── SIGNALS ──────────────────────────────────────────────
Profitability: {s.get('profitability_tier','N/A')} | Cash conversion: {s.get('cash_conversion','N/A')}
Leverage: {s.get('leverage_health','N/A')} | Liquidity: {s.get('liquidity_health','N/A')}
Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'})
Revenue trend: {c.get('trend_revenue','N/A')} | Margin trend: {c.get('trend_margin','N/A')}
Employees: {m.get('employees','N/A')}

── FILING TEXT ──────────────────────────────────────────
MD&A: {(txt.get('mda','') or '')[:800] or '[not available]'}

Risk Factors: {(txt.get('risk_factors','') or '')[:500] or '[not available]'}

Return ONLY a JSON object (no markdown, no text outside JSON):
{{
  "revenue_analysis": "<specific numbers and trend — 1-2 sentences>",
  "profitability": "<margin trajectory and net income — 1-2 sentences>",
  "cash_flow": "<FCF quality and capital allocation — 1-2 sentences>",
  "balance_sheet": "<debt and liquidity — 1 sentence>",
  "key_risks": ["<risk1>", "<risk2>", "<risk3>"],
  "investment_signal": "STRONG_BUY | BUY | HOLD | REDUCE | SELL",
  "signal_rationale": "<one specific sentence with numbers>"
}}"""


def build_multiyear_prompt(ticker: str, company: str, form_type: str,
                            timeline: list[dict]) -> str:
    lines = []
    for entry in timeline:
        p = entry["period"]
        m = entry["metrics"]
        c = entry["changes"]
        rev_yoy = f" ({_pct(c.get('revenue_yoy_pct'))} YoY)" if c.get("revenue_yoy_pct") is not None else ""
        lines.append(
            f"{p}: Rev={_usd(m.get('revenue'))}{rev_yoy} | "
            f"GM={_pct(m.get('gross_margin_pct'))} | NM={_pct(m.get('net_margin_pct'))} | "
            f"FCF={_usd(m.get('free_cash_flow'))} | Trend={c.get('trend_revenue','N/A')} | "
            f"Margin={c.get('trend_margin','N/A')}"
        )

    return f"""You are a senior equity research analyst specializing in financial trend analysis.
Analyze the {len(timeline)}-year financial evolution of {company} ({ticker}) — {form_type} filings.
Period: {timeline[0]['period']} → {timeline[-1]['period']}

FINANCIAL TIMELINE (oldest → newest):
{chr(10).join(lines)}

Return ONLY a JSON object:
{{
  "trend_summary": "<overall revenue and margin trajectory — 2 sentences with specific numbers>",
  "key_inflection_points": [
    "<year: what changed and why — include exact numbers>",
    "<year: next significant shift>"
  ],
  "revenue_cagr": "<calculated CAGR from start to end>",
  "margin_evolution": "<gross and net margin trend with start/end values>",
  "cash_generation_quality": "<FCF trend and capital allocation>",
  "risk_evolution": "<how risk profile changed over the period>",
  "investment_thesis": "<bull case and bear case in 1 sentence each>",
  "overall_signal": "IMPROVING | STABLE | DETERIORATING | MIXED"
}}"""


def _parse_429_wait(err_str: str) -> int:
    m = re.search(r"try again in (\d+)m([\d.]+)s", err_str)
    if m:
        return int(m.group(1)) * 60 + int(float(m.group(2))) + 5
    m = re.search(r"try again in ([\d.]+)s", err_str)
    if m:
        return int(float(m.group(1))) + 5
    return 60


def generate_analysis(prompt: str, llm) -> str:
    for attempt in range(3):
        try:
            text = llm.invoke(prompt).content.strip()
            m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
            if m:
                text = m.group(1)
            json.loads(text)
            return text
        except json.JSONDecodeError:
            return text if text else ""
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                wait = _parse_429_wait(err)
                if wait > LLM_429_MAX_WAIT:
                    logger.warning(f"  429: wait {wait}s > max — skipping")
                    return ""
                logger.warning(f"  429 → sleeping {wait}s (attempt {attempt+1}/3)")
                time.sleep(wait)
                continue
            logger.warning(f"  LLM error: {e}")
            return ""
    return ""


def build_single_pair(record: dict, analysis: str) -> dict:
    m, c, s   = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    has_text  = any(v for v in txt.values() if v and len(v) > 100)
    risks     = [kw for kw in RISK_KW if kw in
                 ((txt.get("risk_factors", "") or "") + (txt.get("mda", "") or "")).lower()][:6]

    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')}\n\n"
        f"── INCOME ──\n"
        f"Revenue: {_usd(m.get('revenue'))} (YoY {_pct(c.get('revenue_yoy_pct'))})\n"
        f"Gross Margin: {m.get('gross_margin_pct','N/A')}% Δ{_pct(c.get('gross_margin_pct_delta'),'pp')}\n"
        f"Net Income: {_usd(m.get('net_income'))} Margin: {m.get('net_margin_pct','N/A')}% "
        f"YoY: {_pct(c.get('net_income_yoy_pct'))}\n"
        f"EPS: {m.get('eps_diluted','N/A')} (YoY {_pct(c.get('eps_yoy_pct'))})\n\n"
        f"── CASH FLOW ──\n"
        f"FCF: {_usd(m.get('free_cash_flow'))} (YoY {_pct(c.get('fcf_yoy_pct'))})\n"
        f"Buybacks: {_usd(m.get('share_repurchases'))} | CapEx: {_usd(m.get('capex'))}\n\n"
        f"── BALANCE SHEET ──\n"
        f"Net Debt: {_usd(m.get('net_debt'))} | D/E: {m.get('debt_to_equity','N/A')} | "
        f"CR: {m.get('current_ratio','N/A')}\n\n"
        f"── SIGNALS ──\n"
        f"{s.get('profitability_tier','N/A')} | "
        f"Rule-of-40: {s.get('rule_of_40_score','N/A')} "
        f"({'PASS' if s.get('rule_of_40_pass') else 'FAIL'}) | "
        f"Trend: {c.get('trend_revenue','N/A')}"
    )
    if has_text:
        inp += f"\n\n── MD&A ──\n{(txt.get('mda','') or '')[:400]}"
        inp += f"\n\n── RISKS ──\n{(txt.get('risk_factors','') or '')[:250]}"
    else:
        inp += "\n\n[Text sections not available]"

    return {
        "instruction": "You are a senior financial analyst. Analyze the SEC filing data and return a structured JSON assessment.",
        "input":       inp,
        "output":      analysis,
        "record_type": "single_period",
        "metadata": {
            "ticker":               meta.get("ticker"),
            "company":              meta.get("company_name"),
            "period":               meta.get("period_of_report"),
            "form_type":            meta.get("form_type"),
            "analysis_mode":        "full" if has_text else "quantitative_only",
            "profitability":        s.get("profitability_tier"),
            "trend_revenue":        c.get("trend_revenue"),
            "trend_margin":         c.get("trend_margin"),
            "risk_keywords":        risks,
            "rule_of_40_pass":      s.get("rule_of_40_pass"),
            "metrics_count":        len(m),
            "text_sections_filled": sum(1 for v in txt.values() if v and len(v) > 100),
        },
    }


def build_multiyear_pair(ticker: str, company: str, form_type: str,
                          timeline: list[dict], analysis: str) -> dict:
    lines = []
    for entry in timeline:
        p, m, c = entry["period"], entry["metrics"], entry["changes"]
        lines.append(
            f"{p}: Rev={_usd(m.get('revenue'))} "
            f"(YoY {_pct(c.get('revenue_yoy_pct'))}) | "
            f"GM={_pct(m.get('gross_margin_pct'))} | "
            f"NM={_pct(m.get('net_margin_pct'))} | "
            f"FCF={_usd(m.get('free_cash_flow'))}"
        )
    inp = (
        f"Analyze the multi-year financial evolution of {company} ({ticker}) "
        f"based on {form_type} filings from "
        f"{timeline[0]['period']} to {timeline[-1]['period']}.\n\n"
        "FINANCIAL TIMELINE (oldest → newest):\n" + "\n".join(lines)
    )
    return {
        "instruction": "You are a senior financial analyst specializing in trend analysis. Analyze this company's multi-year financial evolution and return a structured JSON assessment.",
        "input":       inp,
        "output":      analysis,
        "record_type": "multi_year",
        "metadata": {
            "ticker":       ticker,
            "company":      company,
            "form_type":    form_type,
            "period_start": timeline[0]["period"],
            "period_end":   timeline[-1]["period"],
            "n_periods":    len(timeline),
        },
    }


def build_multiyear_records(
    company_records: list[dict],
    form_type: str,
    ticker: str,
    company_name: str,
    llm,
) -> list[dict]:
    recs = sorted(
        [r for r in company_records
         if r["meta"]["form_type"] == form_type
         and len(r.get("metrics", {})) >= 5],
        key=lambda r: r["meta"]["period_of_report"]
    )
    if len(recs) < 3:
        return []

    def _tl(subset):
        return [{"period": r["meta"]["period_of_report"],
                 "metrics": r["metrics"],
                 "changes": r["changes"],
                 "signals": r["signals"]}
                for r in subset]

    output = []

    if len(recs) >= 4:
        tl       = _tl(recs)
        prompt   = build_multiyear_prompt(ticker, company_name, form_type, tl)
        logger.info(f"  Multi-year {form_type}: {len(recs)}-period window")
        analysis = generate_analysis(prompt, llm)
        if analysis:
            rec_id = hashlib.md5(f"{ticker}_{form_type}_multiyear_{len(recs)}".encode()).hexdigest()[:12]
            output.append({
                "id": rec_id,
                "meta": {"ticker": ticker, "company_name": company_name,
                         "form_type": form_type, "record_type": "multi_year",
                         "period_start": recs[0]["meta"]["period_of_report"],
                         "period_end":   recs[-1]["meta"]["period_of_report"],
                         "n_periods": len(recs)},
                "instruction_pair": build_multiyear_pair(ticker, company_name, form_type, tl, analysis),
            })
            logger.info(f"    ✓ Multi-year ({len(recs)} periods): {len(analysis)} chars")

    for i in range(len(recs) - 2):
        subset   = recs[i:i + 3]
        tl       = _tl(subset)
        start_p  = subset[0]["meta"]["period_of_report"]
        end_p    = subset[-1]["meta"]["period_of_report"]
        prompt   = build_multiyear_prompt(ticker, company_name, form_type, tl)
        analysis = generate_analysis(prompt, llm)
        if analysis:
            rec_id = hashlib.md5(f"{ticker}_{form_type}_3yr_{start_p}_{end_p}".encode()).hexdigest()[:12]
            output.append({
                "id": rec_id,
                "meta": {"ticker": ticker, "company_name": company_name,
                         "form_type": form_type, "record_type": "multi_year_3yr",
                         "period_start": start_p, "period_end": end_p, "n_periods": 3},
                "instruction_pair": build_multiyear_pair(ticker, company_name, form_type, tl, analysis),
            })
            logger.info(f"    ✓ 3yr {start_p}→{end_p}: {len(analysis)} chars")

    return output


# ─── MAIN PIPELINE ────────────────────────────────────────────────────────────

def process_company(
    company:      dict,
    sess:         EDGARSession,
    llm,
    forms:        tuple = ("10-K", "10-Q", "8-K"),
    max_per_form: int   = 8,
    done_accessions: set = None,
) -> list[dict]:
    ticker = company["ticker"]
    logger.info(f"══ {ticker} ({company['name']}) ══")
    done_accessions = done_accessions or set()

    cik = ticker_to_cik(ticker, sess)
    if not cik:
        return []

    xbrl = load_xbrl_facts(cik, sess)
    prev_metrics: dict[str, dict] = {}
    all_records:  list[dict]      = []

    for form_type in forms:
        filings = get_filings(cik, form_type, sess, max_per_form)

        for filing in filings:
            accno  = filing["accession_no"]
            period = filing["period_of_report"]
            prim   = filing["primary_document"]

            # FIX 3: skip by accession, not by ticker
            if accno in done_accessions:
                logger.debug(f"  Skip (already done): {accno}")
                continue

            logger.info(f"  ▶ {form_type} | {period} | {accno}")

            rec_id    = hashlib.md5(f"{ticker}_{form_type}_{period}_{accno}".encode()).hexdigest()[:12]
            base_meta = {
                "ticker":           ticker,
                "company_name":     company["name"],
                "cik":              cik,
                "form_type":        form_type,
                "filed_date":       filing["filed_date"],
                "period_of_report": period,
                "fiscal_year":      int(period[:4]) if period else None,
                "accession_no":     accno,
            }

            if "8-K" in form_type:
                event_text = get_8k_content(cik, accno, prim, sess)
                record     = {
                    "id": rec_id, "meta": base_meta,
                    "metrics": {}, "changes": {}, "signals": {},
                    "text_sections": {"event_text": event_text},
                }
                if event_text and len(event_text) > 300:
                    analysis = generate_analysis(build_single_period_prompt(record), llm)
                    if analysis:
                        record["instruction_pair"] = build_single_pair(record, analysis)
                        logger.info(f"    ✓ 8-K LLM: {len(analysis)} chars")
                all_records.append(record)
                continue

            metrics       = extract_metrics(xbrl, period, form_type) if xbrl else {}
            text_sections = scrape_text_sections(cik, accno, prim, form_type, sess)
            logger.info(f"    Metrics: {len(metrics)} fields")

            prev_key = f"{ticker}_{form_type}"
            changes  = compute_changes(metrics, prev_metrics[prev_key]) if prev_key in prev_metrics else {}
            if metrics:
                prev_metrics[prev_key] = metrics

            signals = compute_signals(metrics, changes)
            record  = {
                "id": rec_id, "meta": base_meta,
                "metrics": metrics, "changes": changes,
                "signals": signals, "text_sections": text_sections,
            }

            if len(metrics) >= 5:
                analysis = generate_analysis(build_single_period_prompt(record), llm)
                if analysis:
                    record["instruction_pair"] = build_single_pair(record, analysis)
                    has_text = any(v for v in text_sections.values() if v and len(v) > 100)
                    mode = "FULL" if has_text else "QUANT"
                    logger.info(f"    ✓ LLM [{mode}]: {len(analysis)} chars")
                else:
                    record["_needs_llm"] = True
                    logger.warning(f"    ✗ LLM skipped — pending retry")
            else:
                logger.warning(f"    ✗ LLM skipped: only {len(metrics)} metrics")

            all_records.append(record)

    for form_type in ("10-K", "10-Q"):
        try:
            multiyear = build_multiyear_records(
                all_records, form_type, ticker, company["name"], llm)
            all_records.extend(multiyear)
            if multiyear:
                logger.success(f"  Multi-year {form_type}: +{len(multiyear)} records")
        except Exception as e:
            logger.warning(f"  Multi-year {form_type} failed: {e}")

    logger.success(f"  {ticker}: {len(all_records)} total records")
    return all_records


def build_dataset(
    tickers:      list[dict],
    output_file:  str   = OUTPUT_FILE,
    forms:        tuple = ("10-K", "10-Q", "8-K"),
    max_per_form: int   = 8,
    resume:       bool  = True,
) -> int:
    llm  = get_llm()
    sess = EDGARSession()

    # FIX 3: Resume by accession_no (not just ticker)
    # Reads all already-processed accession numbers from output file.
    # This allows re-running for a ticker that had some filings missed.
    done_accessions: set[str] = set()
    done_tickers:    set[str] = set()
    if resume and os.path.exists(output_file):
        with open(output_file) as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    accno = rec.get("meta", {}).get("accession_no", "")
                    if accno:
                        done_accessions.add(accno)
                    # Also track tickers with multi-year records (no accession)
                    rt = rec.get("meta", {}).get("record_type", "")
                    if rt in ("multi_year", "multi_year_3yr"):
                        done_tickers.add(rec["meta"]["ticker"])
                except Exception:
                    pass
        if done_accessions:
            logger.info(f"Resuming — {len(done_accessions)} accessions already done")

    logger.info(f"Processing {len(tickers)} companies | forms: {forms}")

    written = 0
    pending = 0

    with open(output_file,  "a", encoding="utf-8") as out, \
         open(PENDING_FILE, "a", encoding="utf-8") as pf:
        for company in tqdm(tickers, desc="Companies"):
            try:
                records = process_company(
                    company=company, sess=sess, llm=llm,
                    forms=forms, max_per_form=max_per_form,
                    done_accessions=done_accessions,
                )
                for rec in records:
                    if rec.pop("_needs_llm", False):
                        pf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        pf.flush()
                        pending += 1
                    else:
                        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        out.flush()
                        written += 1
                logger.info(f"  ✓ {company['ticker']}: {len(records)} records | total: {written}")
            except Exception as e:
                logger.exception(f"  ✗ {company['ticker']} crashed: {e}")
                continue

    if pending > 0 and os.path.exists(PENDING_FILE):
        logger.info(f"\n{pending} pending records — retrying LLM...")
        retried = 0
        llm     = get_llm()
        with open(PENDING_FILE) as pf_in, open(output_file, "a") as out:
            for line in pf_in:
                try:
                    rec      = json.loads(line)
                    analysis = generate_analysis(build_single_period_prompt(rec), llm)
                    if analysis:
                        rec["instruction_pair"] = build_single_pair(rec, analysis)
                        retried += 1
                    out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    out.flush()
                    written += 1
                except Exception as e:
                    logger.warning(f"  Retry failed: {e}")
        os.remove(PENDING_FILE)
        logger.success(f"Retry: {retried}/{pending} completed")

    logger.success(f"Done. {written} total records → {output_file}")
    return written



In [5]:

# ─── ENTRY POINT ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    tickers = get_nasdaq100_tickers()
    tickers = tickers[:3]   # ← remove [:3] for full run

    print("=" * 65)
    print("SEC Fine-Tuning Dataset Builder  v10")
    print("=" * 65)
    print(f"  Companies    : {len(tickers)}")
    print(f"  Forms        : 10-K + 10-Q + 8-K")
    print(f"  Output       : {OUTPUT_FILE}")

    print()
    print("FIXES vs v9:")
    print("  [1] ✓ NetSales added to revenue concept (Apple fix)")
    print("  [2] ✓ Consolidated-only XBRL filter (no segment entries)")
    print("  [3] ✓ Resume by accession_no (not ticker) — AAPL won't be skipped")
    print("  [4] ✓ 8-K XBRL JSON garbage detector")

    print("  [6] ✓ 8-K fin_kw threshold 3→2 (catches more press releases)")
    print("=" * 65)



    count = build_dataset(
        tickers=tickers,
        output_file=OUTPUT_FILE,
        forms=("10-K", "10-Q", "8-K"),
        max_per_form=8,
        resume=True,
    )
    print(f"\nDone: {count} records → {OUTPUT_FILE}")

/tmp/ipykernel_17284/4009438480.py:12: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(
21:20:56 | INFO    | Resuming — 21 accessions already done
21:20:56 | INFO    | Processing 3 companies | forms: ('10-K', '10-Q', '8-K')


SEC Fine-Tuning Dataset Builder  v10
  Companies    : 3
  Forms        : 10-K + 10-Q + 8-K
  Output       : sec_finetune_dataset.jsonl

FIXES vs v9:
  [1] ✓ NetSales added to revenue concept (Apple fix)
  [2] ✓ Consolidated-only XBRL filter (no segment entries)
  [3] ✓ Resume by accession_no (not ticker) — AAPL won't be skipped
  [4] ✓ 8-K XBRL JSON garbage detector
  [6] ✓ 8-K fin_kw threshold 3→2 (catches more press releases)


Companies:   0%|          | 0/3 [00:00<?, ?it/s]21:20:56 | INFO    | ══ AAPL (Apple Inc) ══
21:20:56 | INFO    | CIK table: 10,348 entries
21:20:57 | INFO    |   XBRL: 503 GAAP + 2 DEI
21:20:58 | INFO    |   10-K: 5 filings (oldest→newest)
21:20:58 | INFO    |   10-Q: 8 filings (oldest→newest)
21:20:59 | INFO    |   8-K: 8 filings (oldest→newest)
21:20:59 | SUCCESS |   AAPL: 0 total records
21:20:59 | INFO    |   ✓ AAPL: 0 records | total: 0
Companies:  33%|███▎      | 1/3 [00:03<00:06,  3.36s/it]21:20:59 | INFO    | ══ MSFT (Microsoft Corp) ══
21:21:00 | INFO    |   XBRL: 544 GAAP + 3 DEI
21:21:01 | INFO    |   10-K: 5 filings (oldest→newest)
21:21:01 | INFO    |   ▶ 10-K | 2021-06-30 | 0001564590-21-039151
21:21:05 | INFO    |     Sections: 0/6 filled
21:21:05 | INFO    |     Metrics: 40 fields
21:21:08 | INFO    |     ✓ LLM [QUANT]: 1282 chars
21:21:08 | INFO    |   ▶ 10-K | 2022-06-30 | 0001564590-22-026876
21:21:10 | INFO    |     Sections: 0/6 filled
21:21:10 | INFO    |     Metr


Done: 64 records → sec_finetune_dataset.jsonl
